# Marker gene heatmap for single cell data.
- 24 single cell samples plotted.
- according to the perct_1 annotation for clarity.
- assumes standard_scale = 'var' instead of z-score. (check the last part of the script for explanation).

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# adata = sc.read_h5ad("/path/to/file/annotated_query.h5ad")
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad')

In [ ]:
adata

In [ ]:
set(list(adata.obs['prect_1']))

In [ ]:
# # marker_genes_dict = {
# #   "Actively Dividing cells": ["MKI67", "TOP2A", "HMGB2", "UBE2C", "TYMS"],

# #   "B cells": ["MS4A1", "CD79A", "CD74", "CD37", "CD19"],

# #   "Basal cells": ["KRT5", "KRT14", "TP63", "KRT15", "ITGA6"],

# #   "Endothelial cells": ["PECAM1", "VWF", "KDR", "EMCN", "ENG"],

# #   "Epithelial cells": ["EPCAM", "KRT8", "KRT19", "CLDN4"],

# #   "Fibroblast cells": ["COL1A1", "COL1A2", "DCN", "LUM", "COL3A1"],

# #   "Macrophage cells": ["LST1", "C1QA", "C1QB", "CSF1R", "FCER1G"],

# #   "Mast cells": ["KIT", "CPA3", "MS4A2"],

# #   "Mural cells": ["RGS5", "PDGFRB", "CSPG4", "MCAM", "NOTCH3"],

# #   "Muscle cells": ["ACTA1", "MYH1", "TNNT3", "TNNI1", "CKM"],

# #   "Stromal cells": ["SPARC", "VCAN", "COL5A1"],

# #   "T cells": ["CD3D", "CD3E", "TRAC", "LTB", "IL7R"],
# # }

# marker_genes_dict = {
#   "Actively Dividing cells": ["MKI67", "TOP2A", "HMGB2", "UBE2C", "TYMS", "CENPF"],

#   "B cells": ["MS4A1", "CD79A", "CD74", "CD37", "CD19", "CD79B"], 

#   "Basal cells": ["KRT5", "KRT14", "TP63", "KRT15", "ITGA6"],

#   "Endothelial cells": ["PECAM1", "VWF", "KDR", "EMCN", "ENG"],

#   "Epithelial cells": ["EPCAM", "KRT8", "KRT19", "CLDN4"],

#   "Fibroblast cells": ["COL1A1", "COL1A2", "DCN", "LUM", "COL3A1"],

#   "Macrophage cells": ["LST1", "C1QA", "C1QB", "CSF1R", "FCER1G", "TYROBP"],

#   "Mast cells": ["KIT", "CPA3", "MS4A2"],

#   "Mural cells": ["RGS5", "PDGFRB", "CSPG4", "MCAM", "NOTCH3"],

#   "Muscle cells": ["ACTA1", "MYH1", "TNNT3", "TNNI1", "CKM"],

#   "Stromal cells": ["SPARC", "VCAN", "COL5A1"],

#   "T cells": ["CD3D", "CD3E", "TRAC", "LTB", "IL7R"],
# }

In [ ]:
marker_genes_dict = {
  "Actively Dividing cells": ["MKI67", "TOP2A", "UBE2C"],

  "B cells": ["MS4A1", "CD79A", "CD74"],

  "Basal cells": ["KRT5", "KRT14", "TP63"],

  "Endothelial cells": ["PECAM1", "VWF", "KDR"],

  "Epithelial cells": ["EPCAM", "KRT19", "KRT8"],

  "Fibroblast cells": ["COL1A1", "DCN", "LUM"],

  "Macrophage cells": ["LST1", "C1QA", "TYROBP"],

  "Mast cells": ["KIT", "CPA3", "MS4A2"],

  "Mural cells": ["RGS5", "PDGFRB", "CSPG4"],

  "Muscle cells": ["ACTA1", "TNNT3", "CKM"],

  "Plasma cells": ["MZB1", "IGKC", "JCHAIN"],

  "T cells": ["TRAC", "CD3D", "CD3E"],
}

In [ ]:
# considering a fixed number of cells from every cluster to plot the marker gene heatmap.
n_top_cells = 50

selected_indices = []

for cell_type, genes in marker_genes_dict.items():
    cells = adata.obs_names[adata.obs['prect_1'] == cell_type]

    if len(genes) == 0 or len(cells) == 0:
        continue

    valid_genes = [g for g in genes if g in adata.var_names]
    if not valid_genes:
        continue

    expr = adata[cells, valid_genes].X.toarray() if hasattr(adata.X, 'toarray') else adata[cells, valid_genes].X

    scores = expr.sum(axis=1)

    top_idx = np.argsort(scores)[-n_top_cells:]

    selected_cells = cells[top_idx]
    selected_indices.extend(selected_cells)

adata_top = adata[selected_indices]

# customizing order of cell types, doesn't work most of the time.
# desired_order = ['Epithelial Cell - MYB', 'Basal Cell - MYB', 'CD8+ T Cell', 'Duct Epithelial Cell', 'Endothelial Cell', 'Epithelial Cell', 'Fibroblast Cell', 'Muscle Cell', 'Plasma Cell', 'Stromal Cell', 'Stromal Cell - MYB']

desired_order = [
    "Actively Dividing cells",
    "B cells",
    "Basal cells",
    "Endothelial cells",
    "Epithelial cells",
    "Fibroblast cells",
    "Macrophage cells",
    "Mast cells",
    "Mural cells",
    "Muscle cells",
    "Plasma cells",
    "T cells",
]

adata.obs['prect_1'] = adata.obs['prect_1'].astype('category')
adata.obs['prect_1'] = adata.obs['prect_1'].cat.set_categories(desired_order, ordered=True)

# Plot heatmap using Scanpy's inbuilt heatmap function.

sc.pl.heatmap(
    adata_top,
    marker_genes_dict,
    groupby='prect_1',
    cmap="YlOrRd",
    dendrogram=False,
    show_gene_labels=True,
    swap_axes=True,
    standard_scale='var',  
    show=False,
    figsize=(12, 15)
)

axes = plt.gcf().get_axes()
for ax in axes:
    ax.tick_params(axis='both', labelsize=14)
    # Also increase group/category labels if present
    ax.set_ylabel(ax.get_ylabel(), fontsize=14)
    ax.set_xlabel(ax.get_xlabel(), fontsize=14)

plt.tight_layout()

# plt.savefig(f"{ACC_DATA_ROOT}/outputs/images/marker_gene_heatmap.svg", 
#             dpi = 600, 
#             bbox_inches = 'tight')
# plt.show()
plt.close()


# standard scale = var 
- standardizes the expression of gene in each group compared other groups plotted.
- easier to compare the expression patterns of genes across groups thats what we want to highlight in marker gene heatmap.
- Caveat - the colors don't represent absolute expression values.

In [ ]:
# standard scale = var standardizes the expression of gene in each group compared to 